## Task 1:
Create a catalog cyntexa_dev and a schema sales within it. 

In [0]:
%sql
Create Catalog if not exists cyntexa_dev;

Create schema if not exists cyntexa_dev.sales

## Task 2:
Create a managed table sales.orders_raw with at least 5 columns and insert 10 sample rows. 

In [0]:
%sql
drop table cyntexa_dev.sales.orders_raw

In [0]:
%sql
create table if not exists cyntexa_dev.sales.orders_raw (
    order_id int,
    customer_id string,
    product_id int,
    quantity int,
    status string,
    order_date date
)

In [0]:
%sql
Insert into cyntexa_dev.sales.orders_raw
values
    (1, "CUS1001", 101, 2, 'completed', '2026-01-01'),
    (2, "CUS1002", 102, 1, 'shipped', '2026-01-02'),
    (3, "CUS1001", 103, 3, 'pending', '2026-01-03'),
    (4, "CUS1003", 101, 1, 'completed', '2026-01-04'),
    (5, "CUS1004", 104, 5, 'cancelled', '2026-01-05'),
    (6, "CUS1002", 105, 2, 'shipped', '2026-01-06'),
    (7, "CUS1005", 102, 1, 'completed', '2026-01-07'),
    (8, "CUS1001", 106, 4, 'pending', '2026-01-08'),
    (9, "CUS1004", 103, 2, 'shipped', '2026-01-09'),
    (10, "CUS1006", 107, 1, 'cancelled', '2026-01-10')

In [0]:
%sql
select * from cyntexa_dev.sales.orders_raw

## Task 3:
Create a view sales.orders_view that selects only completed orders. 


In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW cyntexa_dev.sales.orders_view AS
    SELECT * FROM cyntexa_dev.sales.orders_raw
    WHERE status = 'completed'
""")

## Task 4:
(Data Analyst) Explore samples.bakehouse or samples.tpch and run 3 exploratory SELECT queries. 


In [0]:
orders_df = spark.table("samples.tpch.orders")
display(orders_df.limit(10))

In [0]:
top_10_orders_df = (
    orders_df.orderBy(orders_df.o_totalprice, ascending=False)
    .limit(10)
)
display(top_10_orders_df)

In [0]:
from pyspark.sql.functions import *


In [0]:
orders_by_status_df = (
    orders_df.groupBy("o_orderstatus")
    .agg(
        count("*").alias("order_count"),
        sum("o_totalprice").alias("total_revenue")
    )
    .orderBy("total_revenue", ascending=False)
)
display(orders_by_status_df)

Write a SQL UDF that masks the last 4 digits of a customer_id or email column, and apply it in a SELECT against orders_raw.

In [0]:
%sql
Create or Replace function cyntexa_dev.sales.mask_customer_id (customer_id string)
returns string
return Concat(
    left(customer_id, length(customer_id) - 4),
    'XXXX'
)

In [0]:
%sql
Select 
    order_id,
    cyntexa_dev.sales.mask_customer_id(customer_id) as masked_customer_id,
    status,
    order_date
From cyntexa_dev.sales.orders_raw

## Task 6:
Create an external table pointing at a cloud storage/DBFS path and use DESCRIBE EXTENDED to compare its LOCATION and table type against the managed table.

In [0]:
df = spark.table("cyntexa_dev.sales.orders_raw")
df.write.format("delta").mode("overwrite").save("/dbfs:/tmp/cyntexa_dev/orders_external")

I tried to do the saving the table to DBFS but since this is a free edition databricks account I don't have access to other compute than severless which the error says I donno access to.
```
[DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /dbfs:/tmp/cyntexa_dev/orders_external/_delta_log SQLSTATE: 56038
File <command-7678227883787834>, line 2
      1 df = spark.table("cyntexa_dev.sales.orders_raw")
----> 2 df.write.format("delta").mode("overwrite").save("/dbfs:/tmp/cyntexa_dev/orders_external")
```

So now I would go with the volumes the same thing

In [0]:
%sql
Create Volume if not exists cyntexa_dev.sales.external_files

In [0]:
df = spark.table("cyntexa_dev.sales.orders_raw")
df.write.format("delta").mode("overwrite").save("/Volumes/cyntexa_dev/sales/external_files/orders_external")

In [0]:
%sql
Create Table if not exists cyntexa_dev.sales.orders_external
Using Delta
Location '/Volumes/cyntexa_dev/sales/external_files/orders_external'

In [0]:
%sql
Describe Extended cyntexa_dev.sales.orders_raw

As the location need an external url not of the volume, hence the external tables can't be made in the Free Edition

```
[RequestId=c51abe1d-910a-487f-8529-151b2f8f1f52 ErrorClass=INVALID_PARAMETER_VALUE.INVALID_PARAMETER_VALUE] Missing cloud file system scheme
```

## Task 7: 
Data Analyst) Build a second view joining orders_view with a customers table/view and calculate total spend per customer. 

For this task I first made the table for customer, then added the unit_price column which was not there when I first made it then I rerun the previous query to make the order_view again.

Then I joined the view with the table.


In [0]:
%sql
Create table if not exists cyntexa_dev.sales.customers_raw (
    customer_id string,
    customer_name string,
    city string,
    signup_date date
);

Insert Into cyntexa_dev.sales.customers_raw (customer_id, customer_name, city, signup_date) 
values 
    ('CUS1001', 'Kirit_1', 'Jaipur', '2025-01-01'),
    ('CUS1002', 'Kirit_2', 'Mumbai', '2025-02-01'),
    ('CUS1003', 'Kirit_3', 'Dehli', '2025-05-04'),
    ('CUS1004', 'Kirit_4', 'udaipur', '2025-02-12'),
    ('CUS1005', 'Kirit_5', 'Chennai', '2025-07-01'),
    ('CUS1006', 'Kirit_6', 'Bhopal', '2025-10-01')

In [0]:
%sql
Alter Table cyntexa_dev.sales.orders_raw Add column total_price decimal(10, 2);

Update cyntexa_dev.sales.orders_raw set total_price = 670 where total_price is null

In [0]:
%sql
ALTER TABLE cyntexa_dev.sales.orders_raw SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name');

Alter table cyntexa_dev.sales.orders_raw rename column total_price to unit_price

In [0]:
%sql
Create or replace view cyntexa_dev.sales.customer_spend_view as
    select
        c.customer_id,
        c.customer_name,
        c.city,
        sum(o.quantity * o.unit_price) as total_spent,
        count(o.order_id) as total_orders
    from cyntexa_dev.sales.orders_view o
    join cyntexa_dev.sales.customers_raw c
        on o.customer_id = c.customer_id
    group by c.customer_id, c.customer_name, c.city
    order by total_spent desc


In [0]:
%sql
select * from cyntexa_dev.sales.customer_spend_view

## Task 8:
Design a full three-level namespace plan for Cyntexa (catalogs for dev/staging/prod, schemas per business domain) and justify the structure in a short writeup.

In [0]:
%sql
Create catalog if not exists cyntexa_dev;
Create catalog if not exists cyntexa_staging;
Create catalog if not exists cyntexa_prod;

Create Schema if not exists cyntexa_dev.sales;
Create Schema if not exists cyntexa_dev.engineering;
Create Schema if not exists cyntexa_dev.hr;
Create Schema if not exists cyntexa_dev.marketing;
Create Schema if not exists cyntexa_dev.finance;
Create Schema if not exists cyntexa_dev.product;

Create Schema if not exists cyntexa_staging.sales;
Create Schema if not exists cyntexa_staging.engineering;
Create Schema if not exists cyntexa_staging.hr;
Create Schema if not exists cyntexa_staging.marketing;
Create Schema if not exists cyntexa_staging.finance;
Create Schema if not exists cyntexa_staging.product;

Create Schema if not exists cyntexa_prod.sales;
Create Schema if not exists cyntexa_prod.engineering;
Create Schema if not exists cyntexa_prod.hr;
Create Schema if not exists cyntexa_prod.marketing;
Create Schema if not exists cyntexa_prod.finance;
Create Schema if not exists cyntexa_prod.product;

I have made the cyntexa three level design as giving the enviorment at the catalog level and domain in the schemma level.

### Design Justification

catalog is taken as the environment because unity catalog access controls inherited top-down, meaning a permission granted at the catalog level is followed by the schema also and table beneath it.

By making the catalog as a boundary make the permission granted very easy to control, for example if a data engineer have the permmission to write and create in any table in the cyntexa_dev enviorment, he may only have select permission in the cyntexa_prod.

Schemas are used as a bussiness domain here because domains are repeated at schemma level in all the catalogs. The shape of the data remains same in all the catalog which helps in the CI/CD pipeline as it follows the same catalog and schema design.

### Tradeoffs:
The alternative domain at the catalog level and environment at schema level makes the domain level permission simplers, but the domain level permission is harder to control as the broad grant on the "dev" would have to be repeated around all the schemmas individually.

## Task 9:
Write a data-masking strategy: which columns need masking, which role tiers should see unmasked data, and how Unity Catalog permissions would enforce it (this connects forward to Day 8 governance). 


### 1. Columns Requiring Masking

The columns that need the masking are given below in the table
 
| Column | Sensitivity | Reason |
|---|---|---|
| `email` |  High | Can be used to contact or re-identify a person |
| `phone` | High | Direct contact info |
| `customer_id` |  Medium | Identifies a specific customer account |
| `customer_name` | Medium | though lower risk than email/phone since a name alone rarely enables direct contact |

But the **order_id, product_id, quantity, status, order_date, city** should not be masked as they are important fields required when for basic reporting and have a low identification risk.

### 2. Role Tiers and Access
 
| Role | `customer_id` | `email` / `phone` | `customer_name` | 
|---|:---:|:---:|:---:|
| **Data Engineer** | Full | Full | Full |
| **Sales Analyst** | Masked | Masked | Full |
| **Marketing Analyst** | Masked | Masked | Masked |
| **External / Contractor BI user** | Masked | Masked | Masked |
| **Finance / Leadership** | Masked | Masked | Masked |

**The golden princple:** mask everything by default ,unmask when the role's actual job requires the data

- A Data Engineer needs full visablity as they build and debug the pipelines that debug the pipeline which produce this data.
- A sales Analyst requires to know the actuall name of the person but don't need the contact info to do their jobs.
- Others don't need the the actuall info of the customer just the total revenue numbers and such.

### 3. Enforcement in Unity Catalog

Unity catalog enforces this natively by using the columns masks: Column masks control what values a user sees for specific columns. The mask is a SQL UDF that takes the column value as input and returns the original value or a masked version. The return type must match or be castable to the column's data type. Each column can have one mask. Column masks can take other columns as inputs to vary behavior based on multiple attributes.

#### Step 1 — Define the masking function
 
```sql
CREATE OR REPLACE FUNCTION cyntexa_dev.sales.email_mask(email STRING)
RETURNS STRING
RETURN
  CASE
    WHEN is_account_group_member('data_engineers') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@', SPLIT_PART(email, '@', 2))
  END;
```

#### Step 2 — Attach the mask to the column
 
```sql
ALTER TABLE cyntexa_dev.sales.customers
ALTER COLUMN email SET MASK cyntexa_dev.sales.email_mask;
```



## Task 10:
 (Data Analyst) Using samples.tpch, write a query with at least one CTE and one window function to produce a 'top 5 customers by revenue per region' report.

In [0]:
%sql
With customer_revenue as (
    select
        r.r_name as region,
        c.c_custkey as customer_id,
        c.c_name as customer_name,
        sum(l.l_extendedprice * (1 - l.l_discount)) as total_revenue
    from samples.tpch.customer c
    join samples.tpch.orders o on c.c_custkey = o.o_custkey
    join samples.tpch.lineitem l on o.o_orderkey = l.l_orderkey
    join samples.tpch.nation n on c.c_nationkey = n.n_nationkey
    join samples.tpch.region r on n.n_regionkey = r.r_regionkey
    group by r.r_name, c.c_custkey, c.c_name
),
ranked_customers as (
    select 
        region,
        customer_id,
        customer_name,
        total_revenue,
        rank () over (partition by region order by total_revenue desc) as rank
    from customer_revenue
)
select
    region,
    customer_name, 
    total_revenue,
    rank
from ranked_customers
where rank <= 5
order by region, rank;